# Robin transport and transient heat

The scalar RAD local form is skew in advection; its multiplier is $(-K\nabla u+\beta u/2)\cdot n$. The heat integrator uses backward Euler with time-dependent source and Dirichlet data.

The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/457cfab601c84e044e2dec52a7b1c3c4fbc4acd363333060ab259e41baf06b51/08_transport_and_heat-companion.zip"
COMPANION_SHA256 = "457cfab601c84e044e2dec52a7b1c3c4fbc4acd363333060ab259e41baf06b51"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("transport/08_transport_and_heat.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


In [ ]:
import inspect
import numpy as np
from pymhm import TriangleMesh, SkeletonSpace, FaceSpace, assemble
from examples.formulations.scalar import (
    ScalarDiscretization, transport_equations, transport_problem,
    heat_equations, heat_problem, initial_coefficients, recover_scalar,
)

# These declarations produce the blocks consumed by the shared core.
print(inspect.getsource(transport_equations))
print(inspect.getsource(heat_equations))


In [ ]:
mesh = TriangleMesh.unit_square(2)
space = SkeletonSpace(mesh, tuple(FaceSpace.uniform(1) for _ in mesh.faces))

def scalar(points):
    """Return the exact affine scalar field."""
    return 1 + points[:, 0] + 2 * points[:, 1]

data = ScalarDiscretization(mesh, space)
problem = transport_problem(
    data, velocity=(1.0, 0.5), reaction=2.0,
    source=lambda points: 2 * scalar(points) + 2, dirichlet=scalar,
)
system = assemble(problem)
rad = recover_scalar(data, system.solve(), system.local_metadata)
assert rad.l2_error(scalar) < 1e-10

# Each time step declares M/dt + K and the original M*old/dt history load.
heat_data = ScalarDiscretization(mesh, SkeletonSpace(mesh))
previous = initial_coefficients(heat_data, scalar)
times = [0.0, 0.01, 0.03, 0.1]
solutions = []
for old_time, time in zip(times[:-1], times[1:], strict=True):
    problem = heat_problem(
        heat_data, previous, time - old_time,
        source=lambda points: np.ones(len(points)),
        dirichlet=lambda points, time=time: scalar(points) + time,
    )
    system = assemble(problem)
    coefficients = system.solve()
    result = recover_scalar(heat_data, coefficients, system.local_metadata)
    previous = coefficients.fields
    solutions.append(result)
    error = result.l2_error(lambda points, time=time: scalar(points) + time)
    assert error < 1e-10
    print("time", time, "space-time patch error", error)


The stationary local forms are declared in the application, with skew advection and a retained constant amplitude. The velocity here is constant and divergence-free; this example does not declare SUPG or UNUSUAL stabilization.

Each backward-Euler step writes

$$
\left(K\nabla u^{n+1},\nabla v\right)
+\frac{\left(u^{n+1},v\right)}{\Delta t}
+\langle\lambda^{n+1},v\rangle
=\left(f^{n+1},v\right)
+\frac{\left(u^n,v\right)}{\Delta t}.
$$

Backward Euler is first order in time for general data. The affine space-time patch checks signs and history; the automated temporal refinement tests use nonaffine evolution. Both steps call the same variational assembly and solution functions used for other operators.


## Full conservative capacity and SUPG time residual

The full transport application declares the same spatial and temporal
test functions for forcing and previous-state capacity. It supports
one-sided macro coefficients, mixed physical boundary data and exact
local nodal constraints. This affine space-time patch verifies those
declared equations; it is not a literature reproduction.


In [ ]:
from examples.formulations.transient import step_equations, time_load, solve_transport_trajectory

print(inspect.getsource(step_equations))
print(inspect.getsource(time_load))

def capacity(points):
    """Declare a stationary, positive physical capacity."""
    return 1 + 0.2 * points[:, 0]

def forcing(points, time):
    """Independently differentiate rho*u_t-div(grad(u))+div(beta*u)+c*u."""
    return capacity(points) + 0.4 + 0.3 * (scalar(points) + time)

history = solve_transport_trajectory(
    mesh, times, skeleton=space, initial=scalar, capacity=capacity,
    velocity=(0.2, 0.1), reaction=0.3, source=forcing,
    dirichlet=lambda points, time: scalar(points) + time,
    degree=2, local_refinement=4, stabilization="supg",
    dirichlet_enforcement="strong", check_original=True,
)
for time, result in zip(history.times[1:], history.solutions, strict=True):
    assert result.l2_error(lambda points, time=time: scalar(points) + time) < 1e-12
assert np.max(np.abs(history.balance_residuals)) < 1e-12
print({"operator_builds": history.operator_builds, "mass": history.total_mass()})
